In [ ]:
import tkinter as tk

class TicTacToe:
    def __init__(self):
        self.board = [' ' for _ in range(9)]
        self.current_winner = None

    def available_moves(self):
        return [i for i, spot in enumerate(self.board) if spot == ' ']

    def empty_squares(self):
        return ' ' in self.board

    def make_move(self, square, letter):
        if self.board[square] == ' ':
            self.board[square] = letter
            if self.winner(square, letter):
                self.current_winner = letter
            return True
        return False

    def winner(self, square, letter):
        row_ind = square // 3
        row = self.board[row_ind * 3:(row_ind + 1) * 3]
        if all(spot == letter for spot in row):
            return True

        col_ind = square % 3
        column = [self.board[col_ind + i * 3] for i in range(3)]
        if all(spot == letter for spot in column):
            return True

        if square % 2 == 0:
            if all(self.board[i] == letter for i in [0, 4, 8]):
                return True
            if all(self.board[i] == letter for i in [2, 4, 6]):
                return True

        return False


def minimax(state, player):
    max_player = 'X'
    other_player = 'O' if player == 'X' else 'X'

    if state.current_winner:
        if state.current_winner == max_player:
            return {'score': len(state.available_moves()) + 1}
        else:
            return {'score': -(len(state.available_moves()) + 1)}

    if not state.empty_squares():
        return {'score': 0}

    if player == max_player:
        best = {'position': None, 'score': -float('inf')}
    else:
        best = {'position': None, 'score': float('inf')}

    for move in state.available_moves():
        state.make_move(move, player)
        sim_score = minimax(state, other_player)
        state.board[move] = ' '
        state.current_winner = None
        sim_score['position'] = move

        if player == max_player:
            if sim_score['score'] > best['score']:
                best = sim_score
        else:
            if sim_score['score'] < best['score']:
                best = sim_score

    return best


class TicTacToeGUI:
    def __init__(self):
        self.game = TicTacToe()
        self.root = tk.Tk()
        self.root.title("Minimax Tic-Tac-Toe")
        self.root.geometry("450x600")
        self.root.resizable(False, False)
        self.root.eval('tk::PlaceWindow . center')

        main_frame = tk.Frame(self.root, padx=20, pady=5)
        main_frame.pack(expand=True, fill='both')

        title_label = tk.Label(main_frame, text="Minimax AI (X) vs You (O)",
                               font=('Arial', 16, 'bold'))
        title_label.grid(row=0, column=0, columnspan=3, pady=(0, 5))

        board_frame = tk.Frame(main_frame)
        board_frame.grid(row=1, column=0, columnspan=3, pady=(0, 5))

        self.buttons = []
        for i in range(9):
            btn = tk.Button(board_frame, text=' ', font=('Arial', 22, 'bold'),
                            width=5, height=3,
                            command=lambda i=i: self.human_move(i))
            btn.grid(row=i // 3, column=i % 3, padx=8, pady=8)
            self.buttons.append(btn)

        self.status = tk.Label(main_frame, text="Your turn (O)",
                               font=('Arial', 14, 'bold'))
        self.status.grid(row=2, column=0, columnspan=3, pady=(0, 5))

        button_frame = tk.Frame(main_frame)
        button_frame.grid(row=3, column=0, columnspan=3)

        new_game_btn = tk.Button(button_frame, text="New Game",
                                 font=('Arial', 12, 'bold'),
                                 width=12, height=2,
                                 command=self.reset_game)
        new_game_btn.grid(row=0, column=0, padx=15)

        exit_btn = tk.Button(button_frame, text="Exit",
                             font=('Arial', 12, 'bold'),
                             width=12, height=2,
                             command=self.root.quit)
        exit_btn.grid(row=0, column=1)

        self.root.mainloop()

    def human_move(self, pos):
        if self.game.make_move(pos, 'O'):
            self.buttons[pos].config(text='O', state='disabled')

            if self.game.current_winner:
                self.status.config(text="O wins")
                self.disable_all()
            elif not self.game.empty_squares():
                self.status.config(text="Tie game")
                self.disable_all()
            else:
                self.status.config(text="AI thinking")
                self.root.after(500, self.ai_move)

    def ai_move(self):
        if len(self.game.available_moves()) == 9:
            move = 4
        else:
            move = minimax(self.game, 'X')['position']

        self.game.make_move(move, 'X')
        self.buttons[move].config(text='X', state='disabled')

        if self.game.current_winner:
            self.status.config(text="AI wins")
        elif not self.game.empty_squares():
            self.status.config(text="Tie game")
        else:
            self.status.config(text="Your turn (O)")

        if self.game.current_winner or not self.game.empty_squares():
            self.disable_all()

    def disable_all(self):
        for btn in self.buttons:
            btn.config(state='disabled')

    def reset_game(self):
        self.game = TicTacToe()
        for btn in self.buttons:
            btn.config(text=' ', state='normal')
        self.status.config(text="Your turn (O)")


if __name__ == "__main__":
    TicTacToeGUI()
